# 2. Разметка с нуля
Читаем датасет, создаём проект и загружаем картинки с перекрытием 3. Исполнитель рисует все рамки.

Повторный запуск: `ALLOW_TASK_CHANGES = False` разрешает только чтение Яндекс Заданий. Создание, загрузка, модерация и открытие пулов отключены; локальные выгрузки обновляются.

In [1]:
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from seminar.data import read_json, write_json
from seminar.api import request, all_items, upload_tasks, classic_view, boxes_to_annotations

load_dotenv()
Path("results").mkdir(exist_ok=True)

ALLOW_TASK_CHANGES = False

### Датасет

In [2]:
images = read_json("data/dataset.json")
pd.DataFrame(images)[["image_id", "image_path", "n_boxes"]].head()

,image_id,image_path,n_boxes
0,276d0f77ebfd68cc,data/images/276d0f77ebfd68cc.jpg,4
1,2c7fdd64bec8ce82,data/images/2c7fdd64bec8ce82.jpg,0
2,88fb12877b787d10,data/images/88fb12877b787d10.jpg,13
3,a9f8223916d2094f,data/images/a9f8223916d2094f.jpg,14
4,a174779d97951f73,data/images/a174779d97951f73.jpg,9


### Проект
Используем проект 10670 и пул 6756373. Для создания нового проекта и пула задайте оба `EXISTING_*_ID = None`.

In [3]:
project_payload = {
    "public_name": "Выделите весь транспорт на фотографии [MANUAL]",
    "public_description": "Выделите объекты прямоугольниками. Если рамки уже есть, проверьте и исправьте их.",
    "public_instructions": Path("interface/instructions.html").read_text(encoding="utf-8"),
    "task_spec": {
        "input_spec": {
            "image": {"type": "url", "required": True},
            "image_id": {"type": "string", "required": True, "hidden": True},
            "initial_boxes": {"type": "json", "required": True, "hidden": False},
        },
        "output_spec": {
            "result": {"type": "json", "required": True},
            "preannotation_initialized": {"type": "boolean", "required": False},
        },
        "view_spec": classic_view("task"),
    },
    "assignments_issuing_type": "AUTOMATED",
    "assignments_automerge_enabled": False,
}

In [4]:
EXISTING_PROJECT_ID = "10670"
EXISTING_POOL_ID = "6756373"

if EXISTING_POOL_ID and not EXISTING_PROJECT_ID:
    raise ValueError("Для существующего пула укажите EXISTING_PROJECT_ID")

assert EXISTING_PROJECT_ID or ALLOW_TASK_CHANGES, "Укажите существующий проект для повторного запуска"

project = (request("GET", f"projects/{EXISTING_PROJECT_ID}") if EXISTING_PROJECT_ID
           else request("POST", "projects", json=project_payload))
project_id = project["id"]
project_id


'10670'

### Пул

In [5]:
REWARD = 1
OVERLAP = 3

pool_payload = {
    "project_id": project_id,
    "private_name": "manual",
    "may_contain_adult_content": False,
    "reward_per_assignment": REWARD,
    "assignment_max_duration_seconds": 1800,
    "will_expire": (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "auto_accept_solutions": True,
    "defaults": {"default_overlap_for_new_task_suites": OVERLAP},
    "mixer_config": {"real_tasks_count": 1, "golden_tasks_count": 0, "training_tasks_count": 0},
    "filter": {"and": [
        {"category": "computed", "key": "client_type", "operator": "EQ", "value": "BROWSER"},
        {"category": "profile", "key": "languages", "operator": "IN", "value": "RU"},
    ]},
}

In [6]:
assert EXISTING_POOL_ID or ALLOW_TASK_CHANGES, "Укажите существующий пул для повторного запуска"

pool = (request("GET", f"pools/{EXISTING_POOL_ID}") if EXISTING_POOL_ID
        else request("POST", "pools", json=pool_payload))
assert str(pool["project_id"]) == str(project_id), "Пул принадлежит другому проекту"
pool_id = pool["id"]
REWARD = pool["reward_per_assignment"]
OVERLAP = pool["defaults"]["default_overlap_for_new_task_suites"]
pool_id


'6756373'

### Задания

In [7]:
tasks = [
    {"pool_id": pool_id, "overlap": OVERLAP,
     "input_values": {"image": row["image_url"], "image_id": row["image_id"], "initial_boxes": []}}
    for row in images
]
len(tasks), len(tasks) * OVERLAP * REWARD

(60, 180.0)

In [8]:
if EXISTING_POOL_ID:
    print("Задания уже в пуле:", len(all_items("tasks", pool_id=pool_id)))
elif ALLOW_TASK_CHANGES:
    upload_tasks(pool_id, tasks)


Задания уже в пуле: 60


### Модерация

In [9]:
validation_url = f"https://tasks.yandex.ru/api/new/requester/pools/{pool_id}/validate"
validation = request("GET", validation_url)
validation

[{'type': 'HAS_NON_ACTUAL_SNAPSHOT_MAJOR_VERSION',
  'active': False,
  'blocker': False},
 {'type': 'HAS_GS_OR_TRAININGS_WITHOUT_RULE',
  'active': False,
  'blocker': False},
 {'type': 'HAS_RULE_WITHOUT_GS_OR_TRAININGS',
  'active': False,
  'blocker': False},
 {'type': 'HAS_CAPTCHA_WITHOUT_RULE', 'active': False, 'blocker': False},
 {'type': 'HAS_RULE_WITHOUT_CAPTCHA', 'active': False, 'blocker': False},
 {'type': 'HAS_TASK_AND_SUITES_TOGETHER', 'active': False, 'blocker': False},
 {'type': 'HAS_MIXER_AND_SUITES_TOGETHER', 'active': False, 'blocker': False},
 {'type': 'HAS_DYNAMIC_OVERLAP_AND_SUITES_TOGETHER',
  'active': False,
  'blocker': False},
 {'type': 'HAS_DYNAMIC_OVERLAP_WITHOUT_MIXER',
  'active': False,
  'blocker': False},
 {'type': 'HAS_CLOSED_TRAINING', 'active': False, 'blocker': False},
 {'type': 'HAS_EMPTY_QUALITY_CONTROL', 'active': True, 'blocker': False},
 {'type': 'HAS_EMPTY_FILTER', 'active': False, 'blocker': False},
 {'type': 'HAS_MIXER_DEFICIT', 'active': Fa

Если активен `MUST_PASS_MODERATION`, отправим пул на модерацию.

In [10]:
if not ALLOW_TASK_CHANGES:
    print("Повторный запуск: отправка на модерацию отключена")
elif EXISTING_POOL_ID:
    print("Этот пул уже отправлен на модерацию; статус — в предыдущей ячейке.")
elif any(item["type"] == "MUST_PASS_MODERATION" and item.get("active") for item in validation):
    moderation = request(
        "PUT",
        f"https://tasks.yandex.ru/api/new/requester/poolModeration/{pool_id}",
        json={"status": "READY"},
    )
    print(moderation)
else:
    print("Модерация не требуется или уже пройдена")

Повторный запуск: отправка на модерацию отключена


### Запуск
После прохождения модерации выполните следующую ячейку.

In [11]:
if ALLOW_TASK_CHANGES:
    validation = request("GET", validation_url)
    blockers = [item for item in validation if item.get("active") and item.get("blocker", True)]
    if blockers:
        raise ValueError(f"Пул пока нельзя открыть: {blockers}")
    request("POST", f"pools/{pool_id}/open")
else:
    print(f"Повторный запуск: пул {pool_id} не открываем")

Повторный запуск: пул 6756373 не открываем


### Результаты
Забираем результаты.

In [12]:
assignments = all_items("assignments", pool_id=pool_id)
write_json("results/manual.json", assignments)
pd.Series([a["status"] for a in assignments]).value_counts()

ACCEPTED    180
EXPIRED      36
SKIPPED      19
Name: count, dtype: int64